# Computer Vision - Assignment 1


# Question 1

In [ ]:
from os import path, listdir, mkdir

import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
from PIL import Image
from IPython import display

# Uploading files
Q1_DIR_DATA = path.join(".","data","House") # the root of the data files for Q1
Q1_DIR_OUTPUT = "Q1_reconstruction" # the root of the output files for Q1 (e.g. reconstructed 3D points, plots, etc.)
Q1_FILENAMES = {
    "f_P1": "cameraMatrix1.txt",
    "f_P2": "cameraMatrix2.txt",
    "f_house1": "house_1.jpg",
    "f_house2": "house_2.jpg",
    "f_mPoints1": "matchedPoints1.txt",
    "f_mPoints2": "matchedPoints2.txt",
}


files_present = listdir(Q1_DIR_DATA)
for filename in Q1_FILENAMES.values():
  assert filename in files_present

## Task 1

In [ ]:
# Loading data from files
P1 = np.loadtxt(path.join(Q1_DIR_DATA, Q1_FILENAMES["f_P1"]), delimiter=',')
P2 = np.loadtxt(path.join(Q1_DIR_DATA, Q1_FILENAMES["f_P2"]), delimiter=',')
assert P1.shape == P2.shape == (3, 4)

mPoints1 = np.loadtxt(path.join(Q1_DIR_DATA, Q1_FILENAMES["f_mPoints1"]), delimiter=',')
mPoints2 = np.loadtxt(path.join(Q1_DIR_DATA, Q1_FILENAMES["f_mPoints2"]), delimiter=',')
assert mPoints1.shape == mPoints2.shape

house1 = cv.imread(path.join(Q1_DIR_DATA, Q1_FILENAMES["f_house1"]), cv.IMREAD_GRAYSCALE)
house2 = cv.imread(path.join(Q1_DIR_DATA, Q1_FILENAMES["f_house2"]), cv.IMREAD_GRAYSCALE)

# Plotting matches
plt.figure(figsize=(12, 8))
for (i, img, mPoints, title) in [(1, house1, mPoints1, "house_1.jpg"), (2, house2, mPoints2, "house_2.jpg")]:
  X = mPoints[:, 0]
  Y = mPoints[:, 1]

  cmap = plt.get_cmap('rainbow')
  colors = cmap(np.linspace(0, 1, len(mPoints) - 1))
  s = 50
  linewidths = 1
  zorder = 3

  plt.subplot(1, 2, i)
  plt.imshow(img, cmap="gray")

  for i in range(len(mPoints) - 1):
      x_seg = [X[i], X[i+1]]
      y_seg = [Y[i], Y[i+1]]
      color = colors[i]

      plt.plot(x_seg, y_seg, color=color, linewidth=2)

      plt.scatter(X[i], Y[i], color=color, marker='+', s=s, linewidths=linewidths, zorder=zorder)

  plt.scatter(X[-1], Y[-1], color=colors[-1], marker='+', s=s, linewidths=linewidths, zorder=zorder)

  plt.title(title)
  plt.axis('off')


## Task 2

### 2.1. Triangulation
With two matched sets of image points and matrices for both cameras, we can perform triangulation to derive a 3D point cloud in world-coordinates, which we will do by direct linear transformation [lecture 6 (Two-view geometry), slide 17].

In [ ]:
def triangulate(P1, P2, x1, y1, x2, y2):
  p1_1t = P1[0, :]
  p1_2t = P1[1, :]
  p1_3t = P1[2, :]

  p2_1t = P2[0, :]
  p2_2t = P2[1, :]
  p2_3t = P2[2, :]

  a1 = y1*p1_3t - p1_2t
  a2 = p1_1t - x1*p1_3t
  a3 = y2*p2_3t - p2_2t
  a4 = p2_1t - x2*p2_3t
  A = np.array([a1, a2, a3, a4])

  _, _, Vt = np.linalg.svd(A)
  p = Vt[-1]
  p = p / p[-1]
  p = p[:-1]

  return p

points = []
for i in range(len(mPoints1)):
  x1, x2 = mPoints1[i][0], mPoints2[i][0]
  y1, y2 = mPoints1[i][1], mPoints2[i][1]

  p = triangulate(P1, P2, x1, y1, x2, y2)
  points.append(p)

points = np.array(points)

### 2.2. Normalization
We will normalize our point cloud by translating it so that its centroid (center of mass) sits at the world origin.

In [ ]:
centroid = points.mean(axis=0)
points = points - centroid

### 2.3 Projection
To visualize the points in 3D space, we will plot an orthographic wireframe projection of our point cloud on the $xy$ plane.

In [ ]:
def project_xy(points, savepath=None):
  points = points[:, :-1] # Orthographic projection obtained by discarding z-coordinate
  plt.figure(figsize=(12, 8))
  X = points[:, 0]
  Y = points[:, 1]

  cmap = plt.get_cmap('rainbow')
  colors = cmap(np.linspace(0, 1, len(points) - 1))
  s = 50
  linewidths = 1
  zorder = 3

  for i in range(len(points) - 1):
    x_seg = [X[i], X[i+1]]
    y_seg = [Y[i], Y[i+1]]
    color = colors[i]

    plt.plot(x_seg, y_seg, color=color, linewidth=2)
    plt.scatter(X[i], Y[i], color=color, marker='+', s=s, linewidths=linewidths, zorder=zorder)

  plt.scatter(X[-1], Y[-1], color=colors[-1], marker='+', s=s, linewidths=linewidths, zorder=zorder)

  plt.xlim(-4, 4)
  plt.xlabel("x")
  plt.ylim(-4, 4)
  plt.ylabel("y")

  if savepath is None:
    plt.show()
  else:
    plt.savefig(savepath)

project_xy(points)

### 2.4. Initial Rotation
We will perform efficient 3D rotations via [Rodrigues' rotation formula](https://en.wikipedia.org/wiki/Rodrigues%27_rotation_formula), for which OpenCV provides a convenient implementation (`cv2.Rodrigues`); This formula gives a 3D rotation matrix from a 3-element vector (`rvec`) which encodes the axis of rotation in its direction, and the angle of rotation in its magnitude.

In [ ]:
def rotate(points: np.ndarray, rvec: np.ndarray):
  rvec = np.radians(rvec)
  M, _ = cv.Rodrigues(rvec)

  shape_original = points.shape
  rotated = np.reshape((points @ M.T), shape_original)

  return rotated

rng = np.random.default_rng()
rvec = rng.integers(0, high=360, size=3)
points_rotated = rotate(points, rvec)

project_xy(points_rotated)

### 2.5. Animated Reconstruction
We will create an animation by sequentially applying $10°$ rotations to our scene, first about the $x$-axis, and then about the $y$-axis.

In [ ]:
poses = []
# X loop
for _ in range(36):
  rvec = np.array([10, 0, 0], dtype=np.float32)
  points_rotated = rotate(points_rotated, rvec)
  poses.append(np.array(points_rotated))

# Y loop
for _ in range(36):
  rvec = np.array([0, 10, 0], dtype=np.float32)
  points_rotated = rotate(points_rotated, rvec)
  poses.append(np.array(points_rotated))

if not path.isdir(Q1_DIR_OUTPUT):
  mkdir(Q1_DIR_OUTPUT)

filenames_frames = []
for i, points in enumerate(poses):
  savepath = path.join(Q1_DIR_OUTPUT, f"reconstruction_{i}.jpeg")
  filenames_frames.append(savepath)
  project_xy(points, savepath)
  plt.close()

frames = [Image.open(filename) for filename in filenames_frames]
frames[0].save(
    './Q1_reconstruction.gif',
    save_all=True,
    append_images=frames[1:],
    duration=100,
    loop=0
)

display.display(display.Image(filename='./Q1_reconstruction.gif'))

# Q2 infrastructure

In [ ]:
import os
import random
import shutil
import subprocess
from pathlib import Path

import imageio.v2 as imageio
import pandas as pd
from IPython.display import Image as IPyImage, display
from PIL import Image, ImageSequence

PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "Q2"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "Q2"
COLMAP_OUTPUT_ROOT = OUTPUT_ROOT / "colmap_reconstructions"
PLOTS_OUTPUT_ROOT = OUTPUT_ROOT / "plots"
NVS_OUTPUT_ROOT = OUTPUT_ROOT / "novel_view_synthesis"

COLMAP_EXE = Path(os.environ.get("COLMAP_EXE", shutil.which("colmap") or r"C:\Users\fahed\Downloads\colmap-x64-windows-cuda\bin\colmap.exe"))

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
VIDEO_EXTENSIONS = {".mp4"}
GIF_EXTENSIONS = {".gif"}
DEPTH_EXTENSIONS = {".npy", ".npz", ".png", ".tif", ".tiff", ".exr"}

DATASETS = [
    {
        "key": "vase",
        "display_name": "Vase",
        "source_media": DATA_ROOT / "vase.gif",
        "image_dir": DATA_ROOT / "vase_images",
        "database_path": DATA_ROOT / "vase.db",
        "sparse_dir": DATA_ROOT / "vase_colmap_output",
        "matcher": "exhaustive_matcher",
        "depth_dir": None,
    },
    {
        "key": "banana",
        "display_name": "Banana CO3D",
        "source_media": DATA_ROOT / "co3d_banana_images",
        "image_dir": DATA_ROOT / "co3d_banana_images",
        "database_path": DATA_ROOT / "banana.db",
        "sparse_dir": DATA_ROOT / "banana_colmap_output",
        "matcher": "exhaustive_matcher",
        "depth_dir": DATA_ROOT / "co3d_banana_depths",
    },
    {
        "key": "personal",
        "display_name": "Personal 360 Capture",
        "source_media": DATA_ROOT / "my_video.mp4",
        "image_dir": DATA_ROOT / "my_images",
        "database_path": DATA_ROOT / "personal.db",
        "sparse_dir": DATA_ROOT / "personal_colmap_output",
        "matcher": "sequential_matcher",
        "depth_dir": None,
    },
]

RUN_COLMAP_RECONSTRUCTION = True
FORCE_REBUILD_DATABASE = False
FORCE_REBUILD_SPARSE_MODEL = False
MAX_IMAGE_SIZE_FOR_COLMAP = 2000
PNP_SPLITS = [0.90, 0.80, 0.70, 0.50]
PNP_SEEDS = [0, 1, 2, 3, 4]
NVS_K_VALUES = [1, 2, 5, 10, 15]
NVS_TARGET_DATASET_KEY = "banana"
NVS_MAX_IMAGE_SIZE = 1000

for directory in [DATA_ROOT, OUTPUT_ROOT, COLMAP_OUTPUT_ROOT, PLOTS_OUTPUT_ROOT, NVS_OUTPUT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"COLMAP executable: {COLMAP_EXE}")
print("Datasets:")
for dataset in DATASETS:
    print(f"  - {dataset['display_name']}: images={dataset['image_dir']}, matcher={dataset['matcher']}")


# Question 2: Structure from Motion and Novel View Synthesis

This notebook implements the complete pipeline for Question 2:

1. Prepare frame folders from the provided GIF, the selected CO3D object, and the personal 360-degree video.
2. Run sparse Structure-from-Motion using COLMAP.
3. Visualize the recovered camera trajectory and sparse 3D point cloud.
4. Benchmark PnP pose estimation under several train/test split ratios.
5. Synthesize missing CO3D frames for several values of \(k\) using depth-based view warping.

All paths are configured once in the first code cell. The notebook assumes the following project structure:

```text
project_root/
  data/Q2/
    vase.gif
    banana.gif                         # optional, used if available
    myvideo.mp4
    vase_images/
    co3d_banana_images/
    co3d_banana_depths/                # required for Section 2.3
    my_images/
```

If COLMAP is not available through the system `PATH`, set the environment variable `COLMAP_EXE` before running the notebook.


In [ ]:
def check_colmap_available():
    """Check whether the configured COLMAP executable can be called."""
    if not COLMAP_EXE.exists() and shutil.which(str(COLMAP_EXE)) is None:
        print("COLMAP executable was not found. Set COLMAP_EXE or add COLMAP to PATH.")
        return False
    try:
        result = subprocess.run([str(COLMAP_EXE), "help"], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print("COLMAP is available.")
            return True
        print("COLMAP command returned a non-zero status.")
        print(result.stderr[:1000])
        return False
    except Exception as exc:
        print(f"COLMAP check failed: {exc}")
        return False


def check_nvidia_gpu():
    """Print basic NVIDIA GPU information when nvidia-smi is available."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
            capture_output=True,
            text=True,
            timeout=8,
        )
        if result.returncode == 0:
            print("NVIDIA GPU detected:")
            print(result.stdout.strip())
            return True
        print("nvidia-smi was found, but it did not return GPU information.")
        return False
    except FileNotFoundError:
        print("nvidia-smi was not found. COLMAP can still run on CPU or use CUDA if configured separately.")
        return False
    except Exception as exc:
        print(f"GPU check failed: {exc}")
        return False


colmap_ready = check_colmap_available()
gpu_detected = check_nvidia_gpu()
print(f"COLMAP ready: {colmap_ready}")
print(f"GPU detected by nvidia-smi: {gpu_detected}")


## 2.1 Data preparation

The next cell contains top-level helper functions for extracting frames from GIF and video files. The frame folders are created only when the source media exists. Existing frames are kept unless `overwrite=True` is passed.


In [ ]:
def list_images(image_dir):
    """Return sorted image files from a directory."""
    image_dir = Path(image_dir)
    if not image_dir.exists():
        return []
    files = [p for p in image_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
    return sorted(files, key=lambda p: p.as_posix().lower())


def natural_sort_key(path_or_name):
    """Return a key that sorts strings with embedded integers naturally."""
    text = Path(path_or_name).name if not isinstance(path_or_name, str) else path_or_name
    parts = []
    current = ""
    is_digit = False
    for char in text:
        if char.isdigit() != is_digit:
            if current:
                parts.append(int(current) if is_digit else current.lower())
            current = char
            is_digit = char.isdigit()
        else:
            current += char
    if current:
        parts.append(int(current) if is_digit else current.lower())
    return parts


def extract_gif_frames(gif_path, output_dir, overwrite=False):
    """Extract all frames from a GIF into JPEG images."""
    gif_path = Path(gif_path)
    output_dir = Path(output_dir)
    if not gif_path.exists():
        print(f"GIF not found: {gif_path}")
        return 0
    existing = list_images(output_dir)
    if existing and not overwrite:
        print(f"Keeping {len(existing)} existing frames in {output_dir}")
        return len(existing)
    output_dir.mkdir(parents=True, exist_ok=True)
    for old_file in list_images(output_dir):
        old_file.unlink()
    count = 0
    with Image.open(gif_path) as gif_file:
        for frame in ImageSequence.Iterator(gif_file):
            rgb_frame = frame.convert("RGB")
            frame_path = output_dir / f"frame_{count:04d}.jpg"
            rgb_frame.save(frame_path, quality=95)
            count += 1
    print(f"Extracted {count} frames from {gif_path.name} into {output_dir}")
    return count


def extract_video_frames(video_path, output_dir, every_n_frames=1, overwrite=False):
    """Extract frames from a video file into JPEG images."""
    video_path = Path(video_path)
    output_dir = Path(output_dir)
    if not video_path.exists():
        print(f"Video not found: {video_path}")
        return 0
    existing = list_images(output_dir)
    if existing and not overwrite:
        print(f"Keeping {len(existing)} existing frames in {output_dir}")
        return len(existing)
    output_dir.mkdir(parents=True, exist_ok=True)
    for old_file in list_images(output_dir):
        old_file.unlink()
    capture = cv.VideoCapture(str(video_path))
    if not capture.isOpened():
        print(f"Could not open video: {video_path}")
        return 0
    saved_count = 0
    frame_index = 0
    while True:
        ok, frame_bgr = capture.read()
        if not ok:
            break
        if frame_index % every_n_frames == 0:
            frame_path = output_dir / f"frame_{saved_count:04d}.jpg"
            cv.imwrite(str(frame_path), frame_bgr)
            saved_count += 1
        frame_index += 1
    capture.release()
    print(f"Extracted {saved_count} frames from {video_path.name} into {output_dir}")
    return saved_count


def prepare_dataset_frames(dataset, overwrite=False):
    """Prepare the image directory for one dataset when a source media file is available."""
    source_media = dataset["source_media"]
    image_dir = dataset["image_dir"]
    if image_dir.exists() and list_images(image_dir) and not overwrite:
        print(f"{dataset['display_name']}: using existing image folder with {len(list_images(image_dir))} images.")
        return len(list_images(image_dir))
    if source_media.suffix.lower() in GIF_EXTENSIONS:
        return extract_gif_frames(source_media, image_dir, overwrite=overwrite)
    if source_media.suffix.lower() in VIDEO_EXTENSIONS:
        return extract_video_frames(source_media, image_dir, every_n_frames=1, overwrite=overwrite)
    print(f"{dataset['display_name']}: source media is not available or has an unsupported extension.")
    return len(list_images(image_dir))


In [ ]:
frame_counts = {}
for dataset in DATASETS:
    frame_counts[dataset["key"]] = prepare_dataset_frames(dataset, overwrite=False)

pd.DataFrame(
    [{"dataset": dataset["display_name"], "frames": frame_counts[dataset["key"]], "image_dir": str(dataset["image_dir"])} for dataset in DATASETS]
)


## 2.1 COLMAP sparse reconstruction

The reconstruction loop uses:

- `--SiftExtraction.use_gpu 1` for GPU-accelerated SIFT extraction.
- `--SiftMatching.use_gpu 1` for GPU-accelerated matching.
- `--SiftExtraction.max_image_size 2000` to prevent very large images from slowing or failing feature extraction.
- `exhaustive_matcher` for Vase and Banana, because these datasets are manageable and benefit from broad pairwise matching.
- `sequential_matcher` for the personal 360-degree capture, because adjacent frames in a video are strongly related and exhaustive matching scales quadratically with the number of images.


In [ ]:
def run_command(command, description):
    """Run a shell command and raise a clear error if it fails."""
    print(f"\n{description}")
    print(" ".join(str(part) for part in command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-2000:])
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr[-4000:])
        raise RuntimeError(f"Command failed during: {description}")
    return result


def has_sparse_model(sparse_dir):
    """Return True if a COLMAP sparse output folder already contains a model."""
    sparse_dir = Path(sparse_dir)
    if not sparse_dir.exists():
        return False
    for model_dir in sparse_dir.iterdir():
        if model_dir.is_dir() and (model_dir / "images.bin").exists() and (model_dir / "points3D.bin").exists():
            return True
    return False


def run_colmap_reconstruction(dataset):
    """Run feature extraction, matching, and sparse mapping for one dataset."""
    image_dir = dataset["image_dir"]
    database_path = dataset["database_path"]
    sparse_dir = dataset["sparse_dir"]
    matcher = dataset["matcher"]
    image_count = len(list_images(image_dir))
    if image_count == 0:
        print(f"Skipping {dataset['display_name']}: no images found in {image_dir}")
        return False
    sparse_dir.mkdir(parents=True, exist_ok=True)
    if FORCE_REBUILD_DATABASE and database_path.exists():
        database_path.unlink()
    if FORCE_REBUILD_SPARSE_MODEL and sparse_dir.exists():
        shutil.rmtree(sparse_dir)
        sparse_dir.mkdir(parents=True, exist_ok=True)
    if has_sparse_model(sparse_dir) and database_path.exists() and not FORCE_REBUILD_SPARSE_MODEL:
        print(f"Skipping {dataset['display_name']}: existing sparse model found.")
        return True
    feature_command = [
        str(COLMAP_EXE),
        "feature_extractor",
        "--database_path",
        str(database_path),
        "--image_path",
        str(image_dir),
        "--ImageReader.camera_model",
        "SIMPLE_RADIAL",
        "--ImageReader.single_camera",
        "1",
        "--SiftExtraction.use_gpu",
        "1",
        "--SiftExtraction.max_image_size",
        str(MAX_IMAGE_SIZE_FOR_COLMAP),
    ]
    run_command(feature_command, f"Feature extraction for {dataset['display_name']}")
    match_command = [
        str(COLMAP_EXE),
        matcher,
        "--database_path",
        str(database_path),
        "--SiftMatching.use_gpu",
        "1",
    ]
    if matcher == "sequential_matcher":
        match_command.extend(["--SequentialMatching.overlap", "10", "--SequentialMatching.loop_detection", "1"])
    run_command(match_command, f"Feature matching for {dataset['display_name']} using {matcher}")
    mapper_command = [
        str(COLMAP_EXE),
        "mapper",
        "--database_path",
        str(database_path),
        "--image_path",
        str(image_dir),
        "--output_path",
        str(sparse_dir),
    ]
    run_command(mapper_command, f"Sparse mapping for {dataset['display_name']}")
    return has_sparse_model(sparse_dir)


In [ ]:
colmap_status = {}
if RUN_COLMAP_RECONSTRUCTION and colmap_ready:
    for dataset in DATASETS:
        try:
            colmap_status[dataset["key"]] = run_colmap_reconstruction(dataset)
        except Exception as exc:
            colmap_status[dataset["key"]] = False
            print(f"{dataset['display_name']} failed: {exc}")
else:
    for dataset in DATASETS:
        colmap_status[dataset["key"]] = has_sparse_model(dataset["sparse_dir"])
        print(f"{dataset['display_name']}: existing sparse model available = {colmap_status[dataset['key']]}")

pd.DataFrame(
    [{"dataset": dataset["display_name"], "sparse_model_available": colmap_status[dataset["key"]]} for dataset in DATASETS]
)


In [ ]:
def model_quality_score(model_dir):
    """Score a COLMAP model folder by the sizes of its main binary files."""
    model_dir = Path(model_dir)
    images_bin = model_dir / "images.bin"
    points_bin = model_dir / "points3D.bin"
    cameras_bin = model_dir / "cameras.bin"
    score = 0
    for file_path in [images_bin, points_bin, cameras_bin]:
        if file_path.exists():
            score += file_path.stat().st_size
    return score


def find_best_sparse_model(sparse_dir):
    """Find the best numbered COLMAP sparse model subfolder."""
    sparse_dir = Path(sparse_dir)
    if not sparse_dir.exists():
        return None
    candidates = []
    for subfolder in sparse_dir.iterdir():
        if subfolder.is_dir() and ((subfolder / "images.bin").exists() or (subfolder / "images.txt").exists()):
            candidates.append(subfolder)
    if not candidates:
        return None
    return max(candidates, key=model_quality_score)


def convert_sparse_model_to_text(dataset):
    """Convert the selected COLMAP sparse model to text format."""
    model_dir = find_best_sparse_model(dataset["sparse_dir"])
    if model_dir is None:
        print(f"No sparse model found for {dataset['display_name']}")
        return None
    text_dir = model_dir / "txt"
    cameras_txt = text_dir / "cameras.txt"
    images_txt = text_dir / "images.txt"
    points_txt = text_dir / "points3D.txt"
    if cameras_txt.exists() and images_txt.exists() and points_txt.exists():
        print(f"Using existing text model for {dataset['display_name']}: {text_dir}")
        return text_dir
    text_dir.mkdir(parents=True, exist_ok=True)
    command = [
        str(COLMAP_EXE),
        "model_converter",
        "--input_path",
        str(model_dir),
        "--output_path",
        str(text_dir),
        "--output_type",
        "TXT",
    ]
    run_command(command, f"Converting sparse model to TXT for {dataset['display_name']}")
    return text_dir


In [ ]:
for dataset in DATASETS:
    dataset["model_dir"] = find_best_sparse_model(dataset["sparse_dir"])
    if dataset["model_dir"] is not None and colmap_ready:
        dataset["text_model_dir"] = convert_sparse_model_to_text(dataset)
    elif dataset["model_dir"] is not None:
        candidate_text_dir = dataset["model_dir"] / "txt"
        dataset["text_model_dir"] = candidate_text_dir if candidate_text_dir.exists() else dataset["model_dir"]
    else:
        dataset["text_model_dir"] = None

pd.DataFrame(
    [
        {
            "dataset": dataset["display_name"],
            "model_dir": str(dataset.get("model_dir")),
            "text_model_dir": str(dataset.get("text_model_dir")),
        }
        for dataset in DATASETS
    ]
)


In [ ]:
def qvec_to_rotmat(qvec):
    """Convert a COLMAP quaternion into a 3x3 rotation matrix."""
    qw, qx, qy, qz = qvec
    return np.array(
        [
            [1 - 2 * qy * qy - 2 * qz * qz, 2 * qx * qy - 2 * qw * qz, 2 * qx * qz + 2 * qw * qy],
            [2 * qx * qy + 2 * qw * qz, 1 - 2 * qx * qx - 2 * qz * qz, 2 * qy * qz - 2 * qw * qx],
            [2 * qx * qz - 2 * qw * qy, 2 * qy * qz + 2 * qw * qx, 1 - 2 * qx * qx - 2 * qy * qy],
        ],
        dtype=np.float64,
    )


def camera_center_from_pose(R, t):
    """Compute camera center in world coordinates from COLMAP world-to-camera pose."""
    return -R.T @ t


def intrinsic_matrix_from_camera(model, params):
    """Build a camera intrinsic matrix from common COLMAP camera models."""
    params = [float(value) for value in params]
    if model in {"SIMPLE_PINHOLE", "SIMPLE_RADIAL", "SIMPLE_RADIAL_FISHEYE"}:
        f, cx, cy = params[:3]
        return np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]], dtype=np.float64)
    if model in {"PINHOLE", "OPENCV", "OPENCV_FISHEYE", "FULL_OPENCV"}:
        fx, fy, cx, cy = params[:4]
        return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    if model == "RADIAL":
        f, cx, cy = params[:3]
        return np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]], dtype=np.float64)
    raise ValueError(f"Unsupported camera model: {model}")


def parse_cameras_txt(cameras_txt_path):
    """Parse COLMAP cameras.txt."""
    cameras = {}
    cameras_txt_path = Path(cameras_txt_path)
    with cameras_txt_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            camera_id = int(parts[0])
            model = parts[1]
            width = int(parts[2])
            height = int(parts[3])
            params = [float(value) for value in parts[4:]]
            cameras[camera_id] = {
                "model": model,
                "width": width,
                "height": height,
                "params": params,
                "K": intrinsic_matrix_from_camera(model, params),
            }
    return cameras


def parse_images_txt(images_txt_path):
    """Parse COLMAP images.txt including poses and 2D-3D tracks."""
    images = {}
    ordered_names = []
    images_txt_path = Path(images_txt_path)
    lines = images_txt_path.read_text(encoding="utf-8").splitlines()
    line_index = 0
    while line_index < len(lines):
        header = lines[line_index].strip()
        if not header or header.startswith("#"):
            line_index += 1
            continue
        parts = header.split()
        if len(parts) < 10:
            line_index += 1
            continue
        image_id = int(parts[0])
        qvec = np.array([float(value) for value in parts[1:5]], dtype=np.float64)
        tvec = np.array([float(value) for value in parts[5:8]], dtype=np.float64)
        camera_id = int(parts[8])
        image_name = parts[9]
        points_line = ""
        if line_index + 1 < len(lines):
            points_line = lines[line_index + 1].strip()
        values = points_line.split()
        xys = []
        point3d_ids = []
        for point_index in range(0, len(values), 3):
            if point_index + 2 >= len(values):
                break
            xys.append([float(values[point_index]), float(values[point_index + 1])])
            point3d_ids.append(int(values[point_index + 2]))
        R = qvec_to_rotmat(qvec)
        images[image_name] = {
            "image_id": image_id,
            "name": image_name,
            "qvec": qvec,
            "R": R,
            "t": tvec,
            "C": camera_center_from_pose(R, tvec),
            "camera_id": camera_id,
            "xys": np.array(xys, dtype=np.float64),
            "point3D_ids": np.array(point3d_ids, dtype=np.int64),
        }
        ordered_names.append(image_name)
        line_index += 2
    return images, ordered_names


def parse_points3d_txt(points_txt_path):
    """Parse COLMAP points3D.txt."""
    points = {}
    points_txt_path = Path(points_txt_path)
    with points_txt_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            point_id = int(parts[0])
            xyz = np.array([float(parts[1]), float(parts[2]), float(parts[3])], dtype=np.float64)
            rgb = np.array([int(parts[4]), int(parts[5]), int(parts[6])], dtype=np.uint8)
            error = float(parts[7])
            track_values = parts[8:]
            track = []
            for index in range(0, len(track_values), 2):
                if index + 1 < len(track_values):
                    track.append((int(track_values[index]), int(track_values[index + 1])))
            points[point_id] = {"xyz": xyz, "rgb": rgb, "error": error, "track": track}
    return points


def load_colmap_text_model(text_model_dir):
    """Load cameras, images, and points from a COLMAP text model folder."""
    text_model_dir = Path(text_model_dir)
    cameras_path = text_model_dir / "cameras.txt"
    images_path = text_model_dir / "images.txt"
    points_path = text_model_dir / "points3D.txt"
    if not cameras_path.exists() or not images_path.exists() or not points_path.exists():
        raise FileNotFoundError(f"COLMAP text model files are missing in {text_model_dir}")
    cameras = parse_cameras_txt(cameras_path)
    images, ordered_names = parse_images_txt(images_path)
    points = parse_points3d_txt(points_path)
    return cameras, images, points, ordered_names


### COLMAP visualization

For each reconstructed dataset, the next cells plot:

1. A top-view trajectory of camera centers, using the \((x, z)\) coordinates of the camera centers.
2. A sparse 3D point cloud together with camera centers.

These plots verify that the reconstruction follows a circular path around the object.


In [ ]:
def plot_camera_top_view(images, title, output_path=None):
    """Plot a top-view camera trajectory from COLMAP camera centers."""
    if not images:
        print(f"No images to plot for {title}")
        return None
    centers = np.array([image_data["C"] for image_data in images.values()])
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(centers[:, 0], centers[:, 2], marker="o", linewidth=1.5, markersize=3, label="Registered cameras")
    ax.scatter([centers[0, 0]], [centers[0, 2]], marker="^", s=80, label="Start")
    ax.scatter([0], [0], marker="o", s=120, label="Approximate object center")
    ax.set_title(title)
    ax.set_xlabel("World X")
    ax.set_ylabel("World Z")
    ax.axis("equal")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
    plt.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig


def plot_sparse_point_cloud(points, images, title, output_path=None, max_points=15000):
    """Plot a 3D sparse point cloud and recovered camera centers."""
    if not points:
        print(f"No 3D points to plot for {title}")
        return None
    point_items = list(points.values())
    if len(point_items) > max_points:
        indices = np.linspace(0, len(point_items) - 1, max_points).astype(int)
        point_items = [point_items[index] for index in indices]
    xyz = np.array([point["xyz"] for point in point_items])
    rgb = np.array([point["rgb"] for point in point_items], dtype=np.float64) / 255.0
    centers = np.array([image_data["C"] for image_data in images.values()]) if images else np.empty((0, 3))
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=rgb, s=1, alpha=0.7, label="Sparse points")
    if len(centers) > 0:
        ax.plot(centers[:, 0], centers[:, 1], centers[:, 2], marker="o", markersize=2, linewidth=1, label="Camera path")
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.legend()
    plt.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig


In [ ]:
loaded_models = {}
for dataset in DATASETS:
    text_model_dir = dataset.get("text_model_dir")
    if text_model_dir is None:
        print(f"Skipping visualization for {dataset['display_name']}: no text model available.")
        continue
    try:
        cameras, images, points, ordered_names = load_colmap_text_model(text_model_dir)
        loaded_models[dataset["key"]] = {
            "cameras": cameras,
            "images": images,
            "points": points,
            "ordered_names": ordered_names,
        }
        top_view_path = PLOTS_OUTPUT_ROOT / f"{dataset['key']}_top_view.png"
        cloud_path = PLOTS_OUTPUT_ROOT / f"{dataset['key']}_sparse_cloud.png"
        plot_camera_top_view(images, f"{dataset['display_name']} - COLMAP Camera Trajectory Top View", top_view_path)
        plot_sparse_point_cloud(points, images, f"{dataset['display_name']} - Sparse Point Cloud", cloud_path)
    except Exception as exc:
        print(f"Could not visualize {dataset['display_name']}: {exc}")


## 2.2 PnP Benchmark — Practical Fast Evaluation

The assignment asks us to evaluate pose estimation under several train/test splits using 2D–3D correspondences and PnP. A fully strict implementation would rebuild a COLMAP sparse map separately for every dataset, train ratio, and random seed using only the training frames. This is methodologically clean, but it requires many COLMAP reconstructions and is computationally expensive.

For practical runtime, this notebook uses the full COLMAP reconstruction from Section 2.1 as a fixed sparse map and as pseudo-ground truth. For each split, the test images are excluded from the **feature matching and PnP correspondence construction** stage: a test image is localized only by matching its extracted features against features from the train images. The matched train features are then linked to 3D COLMAP points through the train image observations.

**Important limitation.** The sparse 3D points themselves were reconstructed from the full image set, so they may have been influenced by images that later appear in the test split. Therefore, this is a practical approximation rather than a fully strict train-only reconstruction benchmark. The test image pose and its COLMAP 2D observations are not used while estimating the pose; they are used only afterward as pseudo-ground truth for computing errors.

A strict train-only mode was considered and implemented separately, but it is disabled here for runtime reasons:

```python
STRICT_TRAIN_ONLY_COLMAP = False
```

The rotation error is measured by the geodesic angular distance:

$$
e_R(R, \hat{R}) = \cos^{-1}\left(\frac{\operatorname{tr}(R^T \hat{R}) - 1}{2}\right)
$$

The translation error is measured by Euclidean distance between the COLMAP world-to-camera translation vectors:

$$
e_t(t, \hat{t}) = \lVert t - \hat{t} \rVert_2
$$


In [ ]:
# Q2.2 Cell 1 — Fast PnP configuration and feature cache

import pickle
import time
from collections import defaultdict

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

STRICT_TRAIN_ONLY_COLMAP = False  # kept as an explicit methodological switch; disabled for runtime.
REUSE_PNP_FEATURE_CACHE = True

# Set to None for the full required benchmark. For quick debugging, set to a small integer.
MAX_TEST_IMAGES_PER_SPLIT = None

# Keep all datasets if their full COLMAP models were loaded in Q2.1.
ACTIVE_PNP_DATASET_KEYS = ["vase", "banana", "personal"]

# Feature matching / correspondence parameters.
PNP_FEATURE_NFEATURES = 6000
PNP_MAX_REFERENCE_TRAIN_IMAGES = 20
PNP_LOWE_RATIO = 0.75
PNP_MIN_CORRESPONDENCES = 8
PNP_OBSERVATION_DISTANCE_PX = 12.0
PNP_RANSAC_REPROJECTION_ERROR = 8.0
PNP_RANSAC_CONFIDENCE = 0.999
PNP_RANSAC_ITERATIONS = 1500

PNP_FAST_OUTPUT_ROOT = OUTPUT_ROOT / "pnp_fast"
PNP_FEATURE_CACHE_ROOT = PNP_FAST_OUTPUT_ROOT / "feature_cache"
PNP_FAST_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PNP_FEATURE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("STRICT_TRAIN_ONLY_COLMAP:", STRICT_TRAIN_ONLY_COLMAP)
print("Active PnP datasets:", ACTIVE_PNP_DATASET_KEYS)
print("Feature cache root:", PNP_FEATURE_CACHE_ROOT)


def normalize_image_name(name):
    """Return common name variants for robust matching between COLMAP names and disk files."""
    p = Path(str(name))
    return [str(name), p.name, p.stem]


def build_image_index_for_dir(image_dir):
    """Map image names and stems to image paths."""
    index = {}
    for path in list_images(image_dir):
        index[path.name] = path
        index[path.stem] = path
    return index


def find_image_path_for_colmap_name(image_name, image_index):
    """Find the image file corresponding to a COLMAP image name."""
    for key in normalize_image_name(image_name):
        if key in image_index:
            return image_index[key]
    return None


def create_pnp_feature_extractor():
    """Create an OpenCV feature extractor. SIFT is preferred for pose estimation."""
    if hasattr(cv, "SIFT_create"):
        return "SIFT", cv.SIFT_create(nfeatures=PNP_FEATURE_NFEATURES), cv.NORM_L2
    warnings.warn("SIFT is unavailable; falling back to ORB. Results may be less stable.")
    return "ORB", cv.ORB_create(nfeatures=PNP_FEATURE_NFEATURES), cv.NORM_HAMMING


PNP_FEATURE_TYPE, PNP_FEATURE_EXTRACTOR, PNP_MATCH_NORM = create_pnp_feature_extractor()
print("PnP feature extractor:", PNP_FEATURE_TYPE)


def extract_feature_record(image_path):
    """Extract and store keypoint coordinates and descriptors for one image."""
    gray = cv.imread(str(image_path), cv.IMREAD_GRAYSCALE)
    if gray is None:
        return {"points": np.empty((0, 2), dtype=np.float32), "descriptors": None, "shape": None}
    keypoints, descriptors = PNP_FEATURE_EXTRACTOR.detectAndCompute(gray, None)
    if keypoints is None or len(keypoints) == 0 or descriptors is None:
        return {"points": np.empty((0, 2), dtype=np.float32), "descriptors": None, "shape": gray.shape}
    points = np.array([kp.pt for kp in keypoints], dtype=np.float32)
    return {"points": points, "descriptors": descriptors, "shape": gray.shape}


def build_or_load_feature_cache(dataset, model_data):
    """Build/reuse SIFT/ORB feature cache for all registered images of one dataset."""
    dataset_key = dataset["key"]
    cache_path = PNP_FEATURE_CACHE_ROOT / f"{dataset_key}_{PNP_FEATURE_TYPE.lower()}_features.pkl"
    if REUSE_PNP_FEATURE_CACHE and cache_path.exists():
        with cache_path.open("rb") as f:
            cache = pickle.load(f)
        print(f"Loaded feature cache for {dataset['display_name']} from {cache_path}")
        return cache

    image_index = build_image_index_for_dir(dataset["image_dir"])
    cache = {}
    missing = []
    ordered_names = model_data["ordered_names"]
    for image_name in tqdm(ordered_names, desc=f"Extracting {dataset_key} features", unit="img"):
        image_path = find_image_path_for_colmap_name(image_name, image_index)
        if image_path is None:
            missing.append(image_name)
            cache[image_name] = {"points": np.empty((0, 2), dtype=np.float32), "descriptors": None, "shape": None}
            continue
        cache[image_name] = extract_feature_record(image_path)

    with cache_path.open("wb") as f:
        pickle.dump(cache, f)

    if missing:
        print(f"Warning: {len(missing)} registered images were missing on disk for {dataset['display_name']}.")
    print(f"Saved feature cache for {dataset['display_name']} to {cache_path}")
    return cache


In [ ]:
# Q2.2 Cell 2 — 2D–3D correspondence construction and PnP helpers

def rotation_error_degrees(R_estimated, R_ground_truth):
    """Compute angular/geodesic rotation error in degrees."""
    value = (np.trace(R_estimated.T @ R_ground_truth) - 1.0) / 2.0
    value = np.clip(value, -1.0, 1.0)
    return float(np.degrees(np.arccos(value)))


def translation_error(t_estimated, t_ground_truth):
    """Compute Euclidean translation error between COLMAP world-to-camera t-vectors."""
    return float(np.linalg.norm(t_estimated.reshape(3) - t_ground_truth.reshape(3)))


def split_image_names(image_names, train_ratio, seed):
    """Randomly split image names into train and test subsets."""
    image_names = list(image_names)
    rng = random.Random(seed)
    rng.shuffle(image_names)
    train_count = max(1, int(round(train_ratio * len(image_names))))
    train_count = min(train_count, len(image_names) - 1)
    return image_names[:train_count], image_names[train_count:]


def build_observation_spatial_cache(model_data, cell_size=PNP_OBSERVATION_DISTANCE_PX):
    """Build fast lookup from 2D train locations to COLMAP 3D point ids."""
    obs_cache = {}
    for image_name, image_record in model_data["images"].items():
        xys = image_record["xys"]
        point_ids = image_record["point3D_ids"]
        valid = point_ids >= 0
        valid_xys = xys[valid].astype(np.float32)
        valid_point_ids = point_ids[valid].astype(np.int64)

        grid = defaultdict(list)
        for idx, xy in enumerate(valid_xys):
            key = (int(xy[0] // cell_size), int(xy[1] // cell_size))
            grid[key].append(idx)

        obs_cache[image_name] = {
            "xys": valid_xys,
            "point_ids": valid_point_ids,
            "grid": grid,
            "cell_size": float(cell_size),
        }
    return obs_cache


def nearest_colmap_point_id(image_name, xy, obs_cache, max_dist_px=PNP_OBSERVATION_DISTANCE_PX):
    """Find the nearest COLMAP observation to an OpenCV keypoint in a train image."""
    obs = obs_cache.get(image_name)
    if obs is None or len(obs["xys"]) == 0:
        return None, np.inf

    cell_size = obs["cell_size"]
    cx, cy = int(xy[0] // cell_size), int(xy[1] // cell_size)
    candidate_indices = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            candidate_indices.extend(obs["grid"].get((cx + dx, cy + dy), []))

    if not candidate_indices:
        return None, np.inf

    pts = obs["xys"][candidate_indices]
    dists = np.linalg.norm(pts - np.asarray(xy, dtype=np.float32), axis=1)
    best_local = int(np.argmin(dists))
    best_dist = float(dists[best_local])
    if best_dist > max_dist_px:
        return None, best_dist

    best_idx = candidate_indices[best_local]
    return int(obs["point_ids"][best_idx]), best_dist


def select_reference_train_images(test_name, train_names, order_index, max_refs=PNP_MAX_REFERENCE_TRAIN_IMAGES):
    """Select nearby train frames along the circular sequence to reduce matching cost."""
    if max_refs is None or len(train_names) <= max_refs:
        return list(train_names)

    n = max(1, len(order_index))
    test_idx = order_index.get(test_name)
    if test_idx is None:
        return list(train_names)[:max_refs]

    def circular_distance(name):
        idx = order_index.get(name, test_idx)
        d = abs(idx - test_idx)
        return min(d, n - d)

    return sorted(train_names, key=circular_distance)[:max_refs]


def match_test_to_train_3d(test_name, train_names, model_data, feature_cache, obs_cache, order_index):
    """Match one test image against train images and convert train matches to 2D–3D pairs."""
    test_feat = feature_cache.get(test_name)
    if test_feat is None or test_feat["descriptors"] is None or len(test_feat["points"]) == 0:
        return np.empty((0, 2), dtype=np.float64), np.empty((0, 3), dtype=np.float64)

    test_desc = test_feat["descriptors"]
    test_pts = test_feat["points"]
    matcher = cv.BFMatcher(PNP_MATCH_NORM)
    reference_names = select_reference_train_images(test_name, train_names, order_index)

    # Deduplicate by 3D point id and keep the lowest descriptor-distance candidate.
    best_by_point_id = {}

    for train_name in reference_names:
        train_feat = feature_cache.get(train_name)
        if train_feat is None or train_feat["descriptors"] is None or len(train_feat["points"]) == 0:
            continue

        train_desc = train_feat["descriptors"]
        if len(test_desc) < 2 or len(train_desc) < 2:
            continue

        try:
            knn = matcher.knnMatch(test_desc, train_desc, k=2)
        except cv.error:
            continue

        for pair in knn:
            if len(pair) < 2:
                continue
            m, n = pair
            if m.distance >= PNP_LOWE_RATIO * n.distance:
                continue

            test_xy = test_pts[m.queryIdx]
            train_xy = train_feat["points"][m.trainIdx]
            point_id, obs_dist = nearest_colmap_point_id(train_name, train_xy, obs_cache)
            if point_id is None or point_id not in model_data["points"]:
                continue

            score = float(m.distance) + 0.05 * float(obs_dist)
            if point_id not in best_by_point_id or score < best_by_point_id[point_id]["score"]:
                best_by_point_id[point_id] = {
                    "score": score,
                    "image_xy": test_xy.astype(np.float64),
                    "object_xyz": model_data["points"][point_id]["xyz"].astype(np.float64),
                }

    if not best_by_point_id:
        return np.empty((0, 2), dtype=np.float64), np.empty((0, 3), dtype=np.float64)

    pairs = sorted(best_by_point_id.values(), key=lambda item: item["score"])
    image_points = np.array([item["image_xy"] for item in pairs], dtype=np.float64)
    object_points = np.array([item["object_xyz"] for item in pairs], dtype=np.float64)
    return image_points, object_points


def estimate_pose_pnp(image_points, object_points, K):
    """Estimate camera pose with RANSAC PnP."""
    if len(image_points) < PNP_MIN_CORRESPONDENCES or len(object_points) < PNP_MIN_CORRESPONDENCES:
        return None, None, 0

    success, rvec, tvec, inliers = cv.solvePnPRansac(
        object_points.astype(np.float64),
        image_points.astype(np.float64),
        K.astype(np.float64),
        None,
        iterationsCount=PNP_RANSAC_ITERATIONS,
        reprojectionError=PNP_RANSAC_REPROJECTION_ERROR,
        confidence=PNP_RANSAC_CONFIDENCE,
        flags=cv.SOLVEPNP_ITERATIVE,
    )

    if not success or inliers is None or len(inliers) < PNP_MIN_CORRESPONDENCES:
        return None, None, 0

    R_estimated, _ = cv.Rodrigues(rvec)
    return R_estimated, tvec.reshape(3), int(len(inliers))


In [ ]:
# Q2.2 Cell 3 — Run the fast practical PnP benchmark

def evaluate_fast_pnp_for_split(dataset, model_data, feature_cache, obs_cache, train_ratio, seed):
    """Evaluate one dataset/split/seed using train-only feature matching and the full sparse map."""
    ordered_names = list(model_data["ordered_names"])
    if len(ordered_names) < 2:
        return []

    train_names, test_names = split_image_names(ordered_names, train_ratio, seed)
    if MAX_TEST_IMAGES_PER_SPLIT is not None:
        test_names = test_names[:MAX_TEST_IMAGES_PER_SPLIT]

    order_index = {name: idx for idx, name in enumerate(ordered_names)}
    rows = []

    for test_name in tqdm(test_names, desc=f"{dataset['key']} {int(train_ratio*100)}% seed={seed}", leave=False, unit="img"):
        test_record = model_data["images"][test_name]
        K = model_data["cameras"][test_record["camera_id"]]["K"]

        image_points, object_points = match_test_to_train_3d(
            test_name=test_name,
            train_names=train_names,
            model_data=model_data,
            feature_cache=feature_cache,
            obs_cache=obs_cache,
            order_index=order_index,
        )

        R_estimated, t_estimated, inlier_count = estimate_pose_pnp(image_points, object_points, K)
        row = {
            "dataset": dataset["display_name"],
            "image_name": test_name,
            "train_ratio": train_ratio,
            "train_percentage": int(round(train_ratio * 100)),
            "seed": seed,
            "success": R_estimated is not None,
            "correspondences": int(len(image_points)),
            "inliers": int(inlier_count),
            "rotation_error_deg": np.nan,
            "translation_error": np.nan,
        }

        if R_estimated is not None:
            row["rotation_error_deg"] = rotation_error_degrees(R_estimated, test_record["R"])
            row["translation_error"] = translation_error(t_estimated, test_record["t"])

        rows.append(row)

    return rows


pnp_rows = []
pnp_feature_caches = {}
pnp_observation_caches = {}

active_datasets = [dataset for dataset in DATASETS if dataset["key"] in ACTIVE_PNP_DATASET_KEYS]
total_jobs = len(active_datasets) * len(PNP_SPLITS) * len(PNP_SEEDS)
print(f"Total PnP split jobs: {total_jobs}")
print("No train-only COLMAP reconstruction is run in this practical mode.")

job_start = time.time()
job_index = 0

for dataset in active_datasets:
    model_data = loaded_models.get(dataset["key"])
    if model_data is None:
        print(f"Skipping {dataset['display_name']}: full COLMAP model was not loaded in Q2.1.")
        continue

    print("\n" + "=" * 88)
    print(f"Preparing caches for {dataset['display_name']} ({len(model_data['ordered_names'])} registered images)")
    feature_cache = build_or_load_feature_cache(dataset, model_data)
    obs_cache = build_observation_spatial_cache(model_data)

    pnp_feature_caches[dataset["key"]] = feature_cache
    pnp_observation_caches[dataset["key"]] = obs_cache

    for train_ratio in PNP_SPLITS:
        for seed in PNP_SEEDS:
            job_index += 1
            elapsed = time.time() - job_start
            avg = elapsed / max(1, job_index - 1)
            eta = avg * (total_jobs - job_index + 1) if job_index > 1 else np.nan
            eta_msg = f", ETA≈{eta/60:.1f} min" if np.isfinite(eta) else ""

            print(
                f"\n[{job_index}/{total_jobs}] {dataset['display_name']} | "
                f"train={int(round(train_ratio * 100))}% | seed={seed}{eta_msg}"
            )

            split_rows = evaluate_fast_pnp_for_split(
                dataset=dataset,
                model_data=model_data,
                feature_cache=feature_cache,
                obs_cache=obs_cache,
                train_ratio=train_ratio,
                seed=seed,
            )
            pnp_rows.extend(split_rows)

pnp_results = pd.DataFrame(pnp_rows)
if len(pnp_results) > 0:
    pnp_results_path = OUTPUT_ROOT / "pnp_per_image_results.csv"
    pnp_fast_results_path = PNP_FAST_OUTPUT_ROOT / "pnp_fast_per_image_results.csv"
    pnp_results.to_csv(pnp_results_path, index=False)
    pnp_results.to_csv(pnp_fast_results_path, index=False)
    display(pnp_results.head())
    print(f"Saved per-image PnP results to {pnp_results_path}")
    print(f"Saved fast-mode copy to {pnp_fast_results_path}")
else:
    print("No PnP results were produced.")


In [ ]:
# Q2.2 Cell 4 — Aggregate over seeds and generate required plots

def summarize_pnp_results(pnp_results):
    """Summarize PnP results by dataset, train ratio, and seed, then average over seeds."""
    if len(pnp_results) == 0:
        return pd.DataFrame(), pd.DataFrame()

    per_seed_rows = []
    for (dataset_name, train_ratio, seed), group in pnp_results.groupby(["dataset", "train_ratio", "seed"]):
        successful = group[group["success"]].copy()
        per_seed_rows.append(
            {
                "dataset": dataset_name,
                "train_ratio": train_ratio,
                "train_percentage": int(round(train_ratio * 100)),
                "seed": seed,
                "num_test_images": int(len(group)),
                "localized_images": int(len(successful)),
                "success_rate": float(len(successful) / max(1, len(group))),
                "mean_rotation_error_deg": float(successful["rotation_error_deg"].mean()) if len(successful) else np.nan,
                "mean_translation_error": float(successful["translation_error"].mean()) if len(successful) else np.nan,
                "mean_correspondences": float(group["correspondences"].mean()) if len(group) else np.nan,
                "mean_inliers": float(successful["inliers"].mean()) if len(successful) else np.nan,
            }
        )

    per_seed = pd.DataFrame(per_seed_rows)
    if per_seed.empty:
        return per_seed, pd.DataFrame()

    summary = (
        per_seed.groupby(["dataset", "train_ratio", "train_percentage"], as_index=False)
        .agg(
            mean_rotation_error_deg=("mean_rotation_error_deg", "mean"),
            std_rotation_error_deg=("mean_rotation_error_deg", "std"),
            mean_translation_error=("mean_translation_error", "mean"),
            std_translation_error=("mean_translation_error", "std"),
            mean_success_rate=("success_rate", "mean"),
            mean_correspondences=("mean_correspondences", "mean"),
            mean_inliers=("mean_inliers", "mean"),
            mean_localized_images=("localized_images", "mean"),
            mean_num_test_images=("num_test_images", "mean"),
        )
    )

    return per_seed.sort_values(["dataset", "train_percentage", "seed"], ascending=[True, False, True]), summary.sort_values(
        ["dataset", "train_percentage"], ascending=[True, False]
    )


pnp_summary_by_seed, pnp_summary = summarize_pnp_results(pnp_results)

if len(pnp_summary_by_seed) > 0:
    pnp_seed_path = OUTPUT_ROOT / "pnp_summary_by_seed.csv"
    pnp_summary_by_seed.to_csv(pnp_seed_path, index=False)
    display(pnp_summary_by_seed)
    print(f"Saved per-seed PnP summary to {pnp_seed_path}")

if len(pnp_summary) > 0:
    pnp_summary_path = OUTPUT_ROOT / "pnp_summary_results.csv"
    pnp_fast_summary_path = PNP_FAST_OUTPUT_ROOT / "pnp_fast_summary_results.csv"
    pnp_summary.to_csv(pnp_summary_path, index=False)
    pnp_summary.to_csv(pnp_fast_summary_path, index=False)
    display(pnp_summary)
    print(f"Saved PnP summary to {pnp_summary_path}")
else:
    print("PnP summary is empty.")


def plot_pnp_metric(summary, metric_column, ylabel, title, output_path):
    """Plot one PnP metric as a function of the train percentage."""
    if len(summary) == 0:
        print(f"Cannot plot {title}: empty summary.")
        return None

    fig, ax = plt.subplots(figsize=(8, 5))
    for dataset_name, dataset_summary in summary.groupby("dataset"):
        dataset_summary = dataset_summary.sort_values("train_percentage", ascending=False)
        ax.plot(
            dataset_summary["train_percentage"],
            dataset_summary[metric_column],
            marker="o",
            linewidth=2,
            label=dataset_name,
        )

    ax.set_title(title)
    ax.set_xlabel("Train percentage (%)")
    ax.set_ylabel(ylabel)
    ax.set_xticks([90, 80, 70, 50])
    ax.invert_xaxis()
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig


plot_pnp_metric(
    pnp_summary,
    "mean_rotation_error_deg",
    "Mean rotation error (degrees)",
    "PnP Rotation Error vs. Train Percentage",
    PLOTS_OUTPUT_ROOT / "pnp_rotation_error.png",
)

plot_pnp_metric(
    pnp_summary,
    "mean_translation_error",
    "Mean translation error",
    "PnP Translation Error vs. Train Percentage",
    PLOTS_OUTPUT_ROOT / "pnp_translation_error.png",
)

plot_pnp_metric(
    pnp_summary,
    "mean_success_rate",
    "Mean success rate",
    "PnP Success Rate vs. Train Percentage",
    PLOTS_OUTPUT_ROOT / "pnp_success_rate.png",
)


### Answers for Section 2.2

**1. Which type of feature extractor did you use? What would happen if you use a different one?**

I used SIFT features through OpenCV. SIFT is appropriate here because it produces local descriptors that are relatively stable under viewpoint, scale, and illumination changes. If a different extractor were used, the number and reliability of matches would change. A stronger learned feature extractor could improve localization under difficult viewpoint changes, while a weaker or less invariant extractor could reduce the number of correct 2D–3D correspondences and make PnP fail more often.

**2. Which matcher did you use?**

I used a brute-force descriptor matcher with Lowe's ratio test. For each test image, descriptors are matched only against descriptors extracted from train images. To keep runtime reasonable for circular image sequences, I match each test frame mainly against nearby train frames along the image order, since neighboring views have the largest overlap.

**3. How did you get the correspondence between the 2D-3D points?**

The full COLMAP model from Section 2.1 provides a sparse 3D point cloud and, for each train image, 2D observations linked to `POINT3D_ID`s. During each train/test split, I do not use the test image's COLMAP 2D observations. Instead, I extract features in the test image and match them to features in train images. For each matched train feature, I find the nearest COLMAP 2D observation in that train image; if it is close enough and has a valid `POINT3D_ID`, I use the corresponding 3D point from `points3D.txt`. This gives a pair: the 2D test feature location and the matched 3D world point. These pairs are passed to `cv2.solvePnPRansac`.

**Methodological note.** This is a practical approximation designed to keep the runtime manageable. It avoids direct test-frame leakage in the PnP correspondence construction stage, because test poses and test COLMAP observations are not used for estimating the test pose. However, the sparse 3D map itself comes from the full reconstruction, so it may still have been influenced by all frames. A stricter version would rebuild COLMAP from the train images for every split and seed, but that requires many COLMAP reconstructions and is much slower.


## 2.3 Novel View Synthesis: Circular Visualization

For the selected CO3D object, I remove \(k\) consecutive frames and synthesize them using the nearest available frames before and after the gap. The method uses the RGB image, the depth map, the intrinsic matrix, and the COLMAP camera poses.

For a source pixel \(p_s = [u, v, 1]^T\) with depth \(d\), the 3D point in the source camera coordinate system is:

$$
X_s = d K^{-1} p_s
$$

COLMAP stores the camera pose as a world-to-camera transformation:

$$
X_c = R X_w + t
$$

Therefore, the point is transformed from the source camera to world coordinates by:

$$
X_w = R_s^T (X_s - t_s)
$$

Then it is transformed into the target camera:

$$
X_t = R_t X_w + t_t
$$

Finally, it is projected to the target image plane:

$$
\tilde{p}_t = K X_t, \quad u_t = \frac{\tilde{p}_{t,x}}{\tilde{p}_{t,z}}, \quad v_t = \frac{\tilde{p}_{t,y}}{\tilde{p}_{t,z}}
$$

I warp both neighboring source frames into the target pose and blend them. Pixels that project outside the target image or have non-positive depth are ignored. A z-buffer is used during forward warping so that closer projected points overwrite farther ones.


In [ ]:
def get_dataset_by_key(dataset_key):
    """Return a dataset configuration by key."""
    for dataset in DATASETS:
        if dataset["key"] == dataset_key:
            return dataset
    raise KeyError(f"Unknown dataset key: {dataset_key}")


def build_image_path_index(image_dir):
    """Build a lookup table from image names and stems to image paths."""
    index = {}
    for image_path in list_images(image_dir):
        index[image_path.name] = image_path
        index[image_path.stem] = image_path
    return index


def find_depth_directories(dataset):
    """Find likely depth directories for a dataset."""
    candidates = []
    configured_depth_dir = dataset.get("depth_dir")
    if configured_depth_dir is not None and Path(configured_depth_dir).exists():
        candidates.append(Path(configured_depth_dir))
    for path in DATA_ROOT.rglob("*"):
        if path.is_dir() and "depth" in path.name.lower():
            candidates.append(path)
    unique_candidates = []
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved not in seen:
            unique_candidates.append(candidate)
            seen.add(resolved)
    return unique_candidates


def build_depth_path_index(depth_dirs):
    """Build a lookup table from depth file stems to depth paths."""
    index = {}
    for depth_dir in depth_dirs:
        for depth_path in Path(depth_dir).rglob("*"):
            if depth_path.suffix.lower() in DEPTH_EXTENSIONS:
                index[depth_path.name] = depth_path
                index[depth_path.stem] = depth_path
    return index


def find_matching_image_path(image_name, image_index):
    """Find an RGB image path that matches a COLMAP image name."""
    name_path = Path(image_name)
    candidates = [image_name, name_path.name, name_path.stem]
    for candidate in candidates:
        if candidate in image_index:
            return image_index[candidate]
    return None


def find_matching_depth_path(image_name, image_path, depth_index):
    """Find a depth map path that matches standard and CO3D geometric depth names."""
    name_path = Path(image_name)

    base_candidates = [
        image_name,
        name_path.name,
        name_path.stem,
    ]

    if image_path is not None:
        image_path = Path(image_path)
        base_candidates.extend([
            image_path.name,
            image_path.stem,
        ])

    candidates = []
    for candidate in base_candidates:
        candidate = str(candidate)
        candidates.extend([
            candidate,
            candidate + ".geometric.png",
            candidate + ".png",
            candidate + ".jpg",
        ])

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        if candidate not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate)

    for candidate in unique_candidates:
        if candidate in depth_index:
            return depth_index[candidate]

    for candidate in unique_candidates:
        candidate_stem = Path(candidate).stem
        for depth_key, depth_path in depth_index.items():
            if depth_key.startswith(candidate) or depth_key.startswith(candidate_stem):
                return depth_path

    return None


def load_rgb_image(image_path):
    """Load an RGB image from disk."""
    image_bgr = cv.imread(str(image_path), cv.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    return cv.cvtColor(image_bgr, cv.COLOR_BGR2RGB)


def load_depth_map(depth_path):
    """Load a depth map from common CO3D or image formats."""
    depth_path = Path(depth_path)
    suffix = depth_path.suffix.lower()
    if suffix == ".npy":
        depth = np.load(depth_path)
    elif suffix == ".npz":
        data = np.load(depth_path)
        first_key = list(data.keys())[0]
        depth = data[first_key]
    else:
        depth = cv.imread(str(depth_path), cv.IMREAD_UNCHANGED)
        if depth is None:
            raise FileNotFoundError(f"Could not read depth map: {depth_path}")
    depth = np.asarray(depth, dtype=np.float32)
    if depth.ndim == 3:
        depth = depth[:, :, 0]
    finite_depth = depth[np.isfinite(depth)]
    if finite_depth.size > 0 and np.nanmax(finite_depth) > 100.0:
        depth = depth / 1000.0
    depth[~np.isfinite(depth)] = 0.0
    depth[depth < 0] = 0.0
    return depth


def resize_rgb_depth_and_intrinsics(rgb, depth, K, max_size):
    """Resize RGB and depth to a maximum side length while updating intrinsics."""
    height, width = rgb.shape[:2]
    scale = min(1.0, float(max_size) / float(max(height, width)))
    if scale >= 1.0:
        if depth.shape[:2] != (height, width):
            depth = cv.resize(depth, (width, height), interpolation=cv.INTER_NEAREST)
        return rgb, depth, K.copy()
    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))
    rgb_resized = cv.resize(rgb, (new_width, new_height), interpolation=cv.INTER_AREA)
    depth_resized = cv.resize(depth, (new_width, new_height), interpolation=cv.INTER_NEAREST)
    K_resized = K.copy()
    K_resized[0, :] *= scale
    K_resized[1, :] *= scale
    return rgb_resized, depth_resized, K_resized


def forward_warp_with_depth(source_rgb, source_depth, K, source_R, source_t, target_R, target_t):
    """Forward-warp a source RGB-D image into a target camera pose."""
    height, width = source_rgb.shape[:2]
    grid_u, grid_v = np.meshgrid(np.arange(width), np.arange(height))
    u_flat = grid_u.reshape(-1)
    v_flat = grid_v.reshape(-1)
    z_flat = source_depth.reshape(-1)
    colors = source_rgb.reshape(-1, 3)
    valid = z_flat > 1e-6
    if valid.sum() == 0:
        return np.zeros_like(source_rgb), np.zeros((height, width), dtype=np.float32), np.zeros((height, width), dtype=bool)
    u_valid = u_flat[valid]
    v_valid = v_flat[valid]
    z_valid = z_flat[valid]
    colors_valid = colors[valid]
    pixel_homogeneous = np.vstack([u_valid, v_valid, np.ones_like(u_valid)])
    points_source = (np.linalg.inv(K) @ pixel_homogeneous) * z_valid.reshape(1, -1)
    points_world = source_R.T @ (points_source - source_t.reshape(3, 1))
    points_target = target_R @ points_world + target_t.reshape(3, 1)
    target_depth = points_target[2, :]
    valid_target = target_depth > 1e-6
    points_target = points_target[:, valid_target]
    target_depth = target_depth[valid_target]
    colors_valid = colors_valid[valid_target]
    projected = K @ points_target
    projected_u = np.round(projected[0, :] / projected[2, :]).astype(np.int32)
    projected_v = np.round(projected[1, :] / projected[2, :]).astype(np.int32)
    in_bounds = (projected_u >= 0) & (projected_u < width) & (projected_v >= 0) & (projected_v < height)
    projected_u = projected_u[in_bounds]
    projected_v = projected_v[in_bounds]
    target_depth = target_depth[in_bounds]
    colors_valid = colors_valid[in_bounds]
    order = np.argsort(target_depth)[::-1]
    warped = np.zeros_like(source_rgb)
    z_buffer = np.full((height, width), np.inf, dtype=np.float32)
    mask = np.zeros((height, width), dtype=bool)
    for index in order:
        u = projected_u[index]
        v = projected_v[index]
        z_value = target_depth[index]
        if z_value < z_buffer[v, u]:
            z_buffer[v, u] = z_value
            warped[v, u] = colors_valid[index]
            mask[v, u] = True
    z_buffer[~mask] = 0.0
    return warped, z_buffer, mask


def inpaint_missing_pixels(rgb, valid_mask):
    """Inpaint missing pixels in a synthesized image."""
    if valid_mask.all():
        return rgb
    missing_mask = (~valid_mask).astype(np.uint8) * 255
    rgb_bgr = cv.cvtColor(rgb, cv.COLOR_RGB2BGR)
    inpainted_bgr = cv.inpaint(rgb_bgr, missing_mask, 3, cv.INPAINT_TELEA)
    return cv.cvtColor(inpainted_bgr, cv.COLOR_BGR2RGB)


def blend_warped_images(left_rgb, left_depth, left_mask, right_rgb, right_depth, right_mask, alpha):
    """Blend two warped images using gap-relative linear weights."""
    blended = np.zeros_like(left_rgb, dtype=np.float32)
    valid_both = left_mask & right_mask
    valid_left = left_mask & ~right_mask
    valid_right = right_mask & ~left_mask
    blended[valid_left] = left_rgb[valid_left]
    blended[valid_right] = right_rgb[valid_right]
    left_weight = 1.0 - alpha
    right_weight = alpha
    blended[valid_both] = left_weight * left_rgb[valid_both].astype(np.float32) + right_weight * right_rgb[valid_both].astype(np.float32)
    blended = np.clip(blended, 0, 255).astype(np.uint8)
    combined_mask = left_mask | right_mask
    return inpaint_missing_pixels(blended, combined_mask), combined_mask


def add_generated_overlay(rgb, label="Generated"):
    """Add a visible border and label to a generated frame."""
    output = rgb.copy()
    height, width = output.shape[:2]
    cv.rectangle(output, (4, 4), (width - 5, height - 5), (0, 255, 0), max(3, width // 160))
    cv.putText(output, label, (20, 45), cv.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3, cv.LINE_AA)
    return output


def make_triplet_image(original_rgb, generated_rgb):
    """Create a side-by-side original, generated, and absolute-difference triplet."""
    if original_rgb.shape != generated_rgb.shape:
        generated_rgb = cv.resize(generated_rgb, (original_rgb.shape[1], original_rgb.shape[0]), interpolation=cv.INTER_LINEAR)
    difference = cv.absdiff(original_rgb, generated_rgb)
    triplet = np.concatenate([original_rgb, generated_rgb, difference], axis=1)
    height, width = original_rgb.shape[:2]
    labels = [("Original", 10), ("Generated", width + 10), ("Difference", 2 * width + 10)]
    for text, x_position in labels:
        cv.putText(triplet, text, (x_position, height - 20), cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv.LINE_AA)
    return triplet


In [ ]:
def choose_gap_start(ordered_names, k, seed=2026):
    """Choose a reproducible random gap with valid neighbors before and after the removed interval."""
    if len(ordered_names) < k + 2:
        return None
    rng = random.Random(seed + 1000 * int(k))
    return rng.randint(1, len(ordered_names) - k - 1)


def synthesize_target_frame(left_record, right_record, target_record, left_rgb, left_depth, right_rgb, right_depth, K, alpha):
    """Synthesize one target frame from its left and right neighboring frames."""
    left_warped, left_z, left_mask = forward_warp_with_depth(
        left_rgb,
        left_depth,
        K,
        left_record["R"],
        left_record["t"],
        target_record["R"],
        target_record["t"],
    )
    right_warped, right_z, right_mask = forward_warp_with_depth(
        right_rgb,
        right_depth,
        K,
        right_record["R"],
        right_record["t"],
        target_record["R"],
        target_record["t"],
    )
    generated_rgb, generated_mask = blend_warped_images(left_warped, left_z, left_mask, right_warped, right_z, right_mask, alpha)
    return generated_rgb, generated_mask


def prepare_nvs_inputs(dataset):
    """Prepare image, depth, and COLMAP model inputs for novel view synthesis."""
    model_data = loaded_models.get(dataset["key"])
    if model_data is None:
        raise RuntimeError(f"No loaded COLMAP model for {dataset['display_name']}")
    image_index = build_image_path_index(dataset["image_dir"])
    depth_dirs = find_depth_directories(dataset)
    depth_index = build_depth_path_index(depth_dirs)
    if not depth_index:
        raise RuntimeError("No depth maps found. Place CO3D depth files in a folder whose name contains 'depth'.")
    ordered_records = []
    for image_name in model_data["ordered_names"]:
        image_path = find_matching_image_path(image_name, image_index)
        depth_path = find_matching_depth_path(image_name, image_path, depth_index)
        if image_path is None:
            continue
        record = model_data["images"][image_name].copy()
        record["image_path"] = image_path
        record["depth_path"] = depth_path
        ordered_records.append(record)
    if len(ordered_records) < 3:
        raise RuntimeError("Not enough registered images were found for novel view synthesis.")
    return model_data, ordered_records


def load_nvs_source(record, K):
    """Load and resize one RGB-D source frame and update intrinsics."""
    if record["depth_path"] is None:
        raise RuntimeError(f"Missing depth map for {record['name']}")
    rgb = load_rgb_image(record["image_path"])
    depth = load_depth_map(record["depth_path"])
    rgb, depth, K_resized = resize_rgb_depth_and_intrinsics(rgb, depth, K, NVS_MAX_IMAGE_SIZE)
    return rgb, depth, K_resized


def save_rgb_image(path, rgb):
    """Save an RGB image to disk."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    imageio.imwrite(path, rgb)




def make_reconstructed_gif(ordered_records, generated_items, gif_path, fps=12, max_size=512):
    """Create an animated GIF where synthesized frames replace the deleted originals.

    Generated frames are emphasized using a visible border and text label, as required by
    the assignment. The function is self-contained so Section 2.3 does not depend on any
    external helper that may have been removed during notebook merging.
    """
    from PIL import Image as PILImage, ImageDraw

    gif_path = Path(gif_path)
    gif_path.parent.mkdir(parents=True, exist_ok=True)

    generated_by_name = {
        item["target_name"]: Path(item["generated_path"])
        for item in generated_items
    }

    def resize_to_max(rgb, max_side):
        h, w = rgb.shape[:2]
        scale = min(1.0, float(max_side) / float(max(h, w)))
        if scale < 1.0:
            new_w = max(1, int(round(w * scale)))
            new_h = max(1, int(round(h * scale)))
            rgb = cv.resize(rgb, (new_w, new_h), interpolation=cv.INTER_AREA)
        return rgb

    def annotate_generated(rgb, label):
        pil = PILImage.fromarray(rgb.astype(np.uint8))
        draw = ImageDraw.Draw(pil)
        w, h = pil.size
        border = max(4, int(round(min(w, h) * 0.015)))
        # Green border to clearly mark synthesized frames.
        for i in range(border):
            draw.rectangle([i, i, w - 1 - i, h - 1 - i], outline=(0, 255, 0))
        text = label
        pad = 6
        # textbbox is available in modern Pillow; fallback keeps compatibility.
        try:
            bbox = draw.textbbox((0, 0), text)
            text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        except Exception:
            text_w, text_h = draw.textlength(text), 12
        draw.rectangle([0, 0, int(text_w) + 2 * pad, int(text_h) + 2 * pad], fill=(0, 0, 0))
        draw.text((pad, pad), text, fill=(0, 255, 0))
        return np.asarray(pil)

    frames = []
    target_hw = None

    for record in ordered_records:
        name = record["name"]
        is_generated = name in generated_by_name
        frame_path = generated_by_name[name] if is_generated else Path(record["image_path"])

        rgb = load_rgb_image(frame_path)
        rgb = resize_to_max(rgb, max_size)

        if target_hw is None:
            target_hw = rgb.shape[:2]
        elif rgb.shape[:2] != target_hw:
            rgb = cv.resize(rgb, (target_hw[1], target_hw[0]), interpolation=cv.INTER_AREA)

        if is_generated:
            rgb = annotate_generated(rgb, f"Generated k={len(generated_items)}")

        frames.append(rgb.astype(np.uint8))

    if not frames:
        raise RuntimeError("Cannot create GIF because no frames were collected.")

    duration = 1.0 / float(fps)
    imageio.mimsave(gif_path, frames, duration=duration)
    return gif_path


def generate_novel_views_for_k(dataset, k):
    """Remove k consecutive frames and synthesize them at their original poses."""
    model_data, ordered_records = prepare_nvs_inputs(dataset)
    gap_start = choose_gap_start(ordered_records, k)
    if gap_start is None:
        print(f"Cannot generate k={k}: not enough frames.")
        return []
    left_index = gap_start - 1
    right_index = gap_start + k
    left_record = ordered_records[left_index]
    right_record = ordered_records[right_index]
    camera_id = left_record["camera_id"]
    K = model_data["cameras"][camera_id]["K"]
    left_rgb, left_depth, K_resized = load_nvs_source(left_record, K)
    right_rgb, right_depth, _ = load_nvs_source(right_record, K)
    right_rgb = cv.resize(right_rgb, (left_rgb.shape[1], left_rgb.shape[0]), interpolation=cv.INTER_AREA)
    right_depth = cv.resize(right_depth, (left_rgb.shape[1], left_rgb.shape[0]), interpolation=cv.INTER_NEAREST)
    generated_items = []
    output_dir = NVS_OUTPUT_ROOT / f"k_{k}"
    output_dir.mkdir(parents=True, exist_ok=True)
    for offset in range(k):
        target_index = gap_start + offset
        target_record = ordered_records[target_index]
        alpha = (offset + 1) / (k + 1)
        generated_rgb, mask = synthesize_target_frame(
            left_record,
            right_record,
            target_record,
            left_rgb,
            left_depth,
            right_rgb,
            right_depth,
            K_resized,
            alpha,
        )
        generated_rgb = inpaint_missing_pixels(generated_rgb, mask)
        original_rgb = load_rgb_image(target_record["image_path"])
        original_rgb = cv.resize(original_rgb, (generated_rgb.shape[1], generated_rgb.shape[0]), interpolation=cv.INTER_AREA)
        difference = np.abs(original_rgb.astype(np.int16) - generated_rgb.astype(np.int16)).astype(np.uint8)
        generated_path = output_dir / f"reconstructed_k{k}_{Path(target_record['name']).stem}.jpg"
        triplet_path = output_dir / f"triplet_k{k}_{Path(target_record['name']).stem}.jpg"
        save_rgb_image(generated_path, generated_rgb)
        triplet = np.concatenate([original_rgb, generated_rgb, difference], axis=1)
        save_rgb_image(triplet_path, triplet)
        generated_items.append(
            {
                "k": k,
                "target_name": target_record["name"],
                "target_index": target_index,
                "alpha": alpha,
                "generated_path": str(generated_path),
                "triplet_path": str(triplet_path),
                "mask_coverage": float(mask.mean()),
            }
        )
    gif_path = output_dir / f"{dataset['key']}_reconstructed_k{k}.gif"
    make_reconstructed_gif(ordered_records, generated_items, gif_path, fps=12, max_size=NVS_MAX_IMAGE_SIZE)
    for item in generated_items:
        item["gif_path"] = str(gif_path)
    print(f"Generated k={k}: {len(generated_items)} frames, GIF={gif_path}")
    return generated_items


In [ ]:
nvs_dataset = get_dataset_by_key(NVS_TARGET_DATASET_KEY)
nvs_rows = []
for k_value in NVS_K_VALUES:
    try:
        nvs_rows.extend(generate_novel_views_for_k(nvs_dataset, k_value))
    except Exception as exc:
        print(f"Novel view synthesis failed for k={k_value}: {exc}")

nvs_results = pd.DataFrame(nvs_rows)
if len(nvs_results) > 0:
    nvs_results_path = NVS_OUTPUT_ROOT / "nvs_results.csv"
    nvs_results.to_csv(nvs_results_path, index=False)
    display(nvs_results)
    print(f"Saved NVS results to {nvs_results_path}")
else:
    print("No novel view synthesis results were produced.")


In [ ]:
def display_nvs_examples(nvs_results):
    """Display triplets and reconstructed GIFs for all evaluated k values."""
    if len(nvs_results) == 0:
        print("No NVS examples to display.")
        return

    required_k_values = [1, 2, 5, 10, 15]

    print("Displaying original / generated / absolute-difference triplets for all k values.")
    for k_value in required_k_values:
        subset = nvs_results[nvs_results["k"] == k_value]
        if len(subset) == 0:
            print(f"No triplet result found for k={k_value}.")
            continue

        # Use the middle generated frame for this k value, because it is usually the hardest case.
        triplet_path = Path(subset.iloc[len(subset) // 2]["triplet_path"])

        if triplet_path.exists():
            triplet_rgb = imageio.imread(triplet_path)
            plt.figure(figsize=(14, 5))
            plt.imshow(triplet_rgb)
            plt.axis("off")
            plt.title(f"k={k_value}: original frame, generated frame, and absolute difference")
            plt.show()
        else:
            print(f"Triplet file missing for k={k_value}: {triplet_path}")

    print("Displaying reconstructed GIFs for all k values.")
    for k_value in required_k_values:
        subset = nvs_results[nvs_results["k"] == k_value]
        if len(subset) == 0:
            print(f"No GIF result found for k={k_value}.")
            continue

        gif_path = Path(subset.iloc[0]["gif_path"])

        if gif_path.exists():
            print(f"Reconstructed circular GIF for k={k_value}: {gif_path}")
            display(IPyImage(filename=str(gif_path)))
        else:
            print(f"GIF file missing for k={k_value}: {gif_path}")


display_nvs_examples(nvs_results)

### Answers for Section 2.3

**1. How do the generated frames differ for high values of \(k\) when the synthesized frame lies in the middle of the deleted sequence versus near its edges?**

For high values of \(k\), the frames near the edges of the deleted interval are usually better. They are close to one of the available source frames, so the viewpoint change is small and the depth-based warp remains stable. Frames in the middle of a long deleted interval are harder to synthesize because they are far from both source views. This increases disocclusion, hole regions, stretching artifacts, and texture ambiguity. As a result, middle frames for \(k=10\) or \(k=15\) typically look blurrier and less geometrically consistent than edge frames.

**2. How is the depth map used in the solution?**

The depth map gives the distance from the camera to each source pixel. For each source pixel $$([u, v, 1]^T)$$, I back-project it into 3D using $$(X_s = dK^{-1}p_s)$$. Then I transform it to world coordinates using $$(X_w = R_s^T(X_s - t_s))$$, transform it into the target camera using $$(X_t = R_tX_w + t_t)$$, and project it with $$(\tilde{p}_t = KX_t)$$. The pixel color is written into the target image at $$((u_t, v_t))$$. A z-buffer keeps the closest projected point when several source pixels map to the same target pixel.

**3. How would the solution change if the depth map were not provided?**

Without depth, each pixel cannot be placed at a known 3D position. The current back-projection step would be impossible. A replacement solution would need to estimate depth first, for example from COLMAP sparse points, multi-view stereo, optical flow with triangulation, or a monocular depth network. Another simpler but less accurate option would be 2D optical-flow interpolation between neighboring frames. That approach can work for small gaps but does not explicitly handle 3D parallax, so it usually degrades faster as \(k\) increases.

### Critical discussion

As the train percentage decreases from 90% to 50%, the train map contains fewer 3D points. This reduces the number of valid 2D-3D correspondences available for PnP, so both rotation and translation errors are expected to increase. CO3D data is usually cleaner because the object is centered and the circular motion is more controlled. Personal capture is more likely to contain motion blur, uneven lighting, imperfect circular motion, and background clutter, so its matches and recovered poses are usually less stable.


## Submission checklist

The notebook produces the required items for Question 2:

- Sparse COLMAP reconstruction for Vase, CO3D Banana, and Personal Capture.
- Top-view camera trajectory plots.
- Sparse 3D point cloud plots.
- Practical PnP benchmark for train ratios 90%, 80%, 70%, and 50%, using train-only feature matching against the full Q2.1 sparse map.
- Five random seeds for every split.
- Mean rotation and translation error tables.
- Rotation and translation error plots.
- Depth-based novel view synthesis for \(k=1,2,5,10,15\).
- GIF outputs that mark generated frames with a visible border and label.
- Triplet images showing original, generated, and difference images.
- Written methodology, mathematical explanation, and discussion.
